# BigQuery TPC-DS 1G — View Layer Builder

Creates a clean `mstr_view` dataset in the `mstr-tpc` BigQuery project with one view per TPC-DS table.  
The MSTR project will connect to these views instead of the raw TPC-DS tables.

In [ ]:
import json
import yaml
from pprint import pprint
from google.cloud import bigquery
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from mstr_robotics.bq_connector import get_bq_client, get_bq_config

## Config

In [ ]:
# All settings live in config/bq_config.yml
cfg = get_bq_config()

BQ_PROJECT   = cfg["project"]        # mstr-tpc  — where views are written
SRC_PROJECT  = cfg["src_project"]    # bigquery-public-data — where raw tables live
SRC_DATASET  = cfg["src_dataset"]    # tpc_ds_1g
VIEW_DATASET = cfg["view_dataset"]   # mstr_view
LOCATION     = cfg.get("location", "US")

print(f"view project : {BQ_PROJECT}")
print(f"src  project : {SRC_PROJECT}")
print(f"src  dataset : {SRC_DATASET}")
print(f"view dataset : {VIEW_DATASET}")
print(f"location     : {LOCATION}")

## Connect to BigQuery

In [ ]:
# get_bq_client() tries:
#   1. config/bq_sa_key.json  (service account)
#   2. config/bq_user_token.json (cached browser token)
#   3. raises a clear error with setup instructions
#
# First time on a machine without a SA key → run login_browser() once instead:
#   from mstr_robotics.bq_connector import login_browser
#   bq = login_browser()   # opens browser, caches token, then get_bq_client() works forever

from mstr_robotics.bq_connector import get_bq_client, login_browser

bq = login_browser()
print(f"Client project: {bq.project}")

## Upload MSTR tutorial tables (for MSTR project builder)

In [ ]:
import pandas as pd
from google.cloud import bigquery
from pathlib import Path

CSV_DIR   = Path(r"C:\Users\danie\OneDrive\Dokumente\MSTR_ROBOTICS\datasets\tutorial_tables")
TARGET_DS = f"{BQ_PROJECT}.{VIEW_DATASET}"   # mstr-tpc.mstr_view

# ── 1. Ensure dataset exists ──────────────────────────────────────────────────
ds = bigquery.Dataset(TARGET_DS)
ds.location = LOCATION
bq.create_dataset(ds, exists_ok=True)
print(f"Dataset ready: {TARGET_DS}")

# ── 2. Drop all existing tables/views ────────────────────────────────────────
existing = list(bq.list_tables(TARGET_DS))
print(f"Dropping {len(existing)} existing objects ...")
for tbl in existing:
    bq.delete_table(tbl.reference, not_found_ok=True)
    print(f"  dropped  {tbl.table_id}")

# ── 3. Upload each CSV via pandas (autodetect types from data) ────────────────
csv_files = sorted([f for f in CSV_DIR.glob("*.csv") if f.name != "table_catalog.csv"])
print(f"\nUploading {len(csv_files)} CSV files → {TARGET_DS} ...")

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    encoding="UTF-8",
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
)

results = []
for csv_path in csv_files:
    table_name = csv_path.stem
    table_ref  = f"{TARGET_DS}.{table_name}"

    try:
        with open(csv_path, "rb") as f:
            job = bq.load_table_from_file(f, table_ref, job_config=job_config)
        job.result()
        rows = bq.get_table(table_ref).num_rows
        results.append({"table": table_name, "status": "OK", "rows": rows})
        print(f"  ✓  {table_name:<40}  {rows:>8,} rows")
    except Exception as e:
        results.append({"table": table_name, "status": "ERROR", "error": str(e)})
        print(f"  ✗  {table_name}: {e}")

ok  = [r for r in results if r["status"] == "OK"]
err = [r for r in results if r["status"] == "ERROR"]
print(f"\nDone — {len(ok)} tables uploaded, {len(err)} errors.")
if err:
    for e in err:
        print(f"  ERROR: {e['table']}: {e['error']}")


## Load mstr_view tables into Preset via REST API

In [ ]:
import requests

# ── Preset credentials ────────────────────────────────────────────────────────
# Get these from https://manage.app.preset.io/app/user → API Keys
PRESET_TOKEN  = "YOUR_PRESET_API_TOKEN"    # API token name
PRESET_SECRET = "YOUR_PRESET_API_SECRET"   # API token secret

# ── Step 1: get a short-lived JWT ─────────────────────────────────────────────
auth_resp = requests.post(
    "https://api.app.preset.io/v1/auth/",
    json={"name": PRESET_TOKEN, "secret": PRESET_SECRET},
)
auth_resp.raise_for_status()
jwt = auth_resp.json()["payload"]["access_token"]
headers = {"Authorization": f"Bearer {jwt}", "Content-Type": "application/json"}
print("✓ authenticated to Preset")

# ── Step 2: find your workspace ───────────────────────────────────────────────
teams_resp = requests.get("https://api.app.preset.io/v1/teams/", headers=headers)
teams_resp.raise_for_status()
teams = teams_resp.json()["payload"]
for t in teams:
    print(f"  team: {t['name']}  slug={t['name']}")
    for ws in t.get("workspaces", []):
        print(f"    workspace: {ws['title']}  hostname={ws['hostname']}")

In [ ]:
# ── Step 3: set workspace hostname + find BigQuery database ID ────────────────
# Copy the hostname from the output above  (e.g. "abc123.us1a.app.preset.io")
WS_HOST = "YOUR_WORKSPACE_HOSTNAME"   # e.g. "abc123.us1a.app.preset.io"

db_resp = requests.get(f"https://{WS_HOST}/api/v1/database/", headers=headers)
db_resp.raise_for_status()
databases = db_resp.json()["result"]
for db in databases:
    print(f"  id={db['id']}  name={db['database_name']}  backend={db['backend']}")

In [ ]:
# ── Step 4: create one dataset per table ─────────────────────────────────────
# Copy the BigQuery database id from the output above
BQ_DATABASE_ID = 1   # ← replace with actual id

TABLES = [
    "order_detail",
    "lu_employee",
    "lu_day",
    "lu_item",
    "lu_subcateg",
    "lu_call_ctr",
    "lu_month",
    "lu_region",
    "lu_category",
]

BQ_SCHEMA  = "mstr_view"     # BigQuery dataset name
BQ_PROJECT = "mstr-tpc"      # BigQuery project (used as catalog in Superset)

results = []
for table in TABLES:
    payload = {
        "database": BQ_DATABASE_ID,
        "schema":   BQ_SCHEMA,
        "table_name": table,
        "catalog":  BQ_PROJECT,   # BigQuery project acts as catalog
    }
    resp = requests.post(
        f"https://{WS_HOST}/api/v1/dataset/",
        headers=headers,
        json=payload,
    )
    if resp.status_code in (200, 201):
        ds_id = resp.json()["id"]
        results.append({"table": table, "status": "created", "id": ds_id})
        print(f"  ✓  {table:<30}  dataset_id={ds_id}")
    elif resp.status_code == 422 and "already exists" in resp.text:
        results.append({"table": table, "status": "already exists"})
        print(f"  –  {table:<30}  already exists")
    else:
        results.append({"table": table, "status": "ERROR", "body": resp.text})
        print(f"  ✗  {table:<30}  {resp.status_code}: {resp.text[:120]}")

ok  = [r for r in results if r["status"] in ("created", "already exists")]
err = [r for r in results if r["status"] == "ERROR"]
print(f"\nDone — {len(ok)} OK, {len(err)} errors.")